In [1]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import csv
import json
import glob
import math
import time

import numpy as np
import pandas as pd
import regex as re
import torch

from tqdm import tqdm

from datasets import load_dataset, Dataset
from huggingface_hub import login

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)


# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

!pip install regex
!pip install datasets
!pip install transformers datasets accelerate scikit-learn pandas tqdm

In [ ]:
# ============================================================
# DOWNLOAD AND EXTRACT RAW OPENSUBTITLES DATA
# ============================================================

!wget -P /content/drive/MyDrive/code_switch_project/data/raw \
https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/mono/de.txt.gz

!gunzip /content/drive/MyDrive/code_switch_project/data/raw/de.txt.gz

!wget -P /content/drive/MyDrive/code_switch_project/data/raw \
https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/mono/fr.txt.gz

!gunzip /content/drive/MyDrive/code_switch_project/data/raw/fr.txt.gz

!ls -lah /content/drive/MyDrive/code_switch_project/data/raw

In [ ]:
# ============================================================
# INSPECT RAW OPENSUBTITLES DATA
# ============================================================

file_path = "/content/drive/MyDrive/code_switch_project/data/raw/de.txt"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for _ in range(30):
        print(next(f))


file_path = "/content/drive/MyDrive/code_switch_project/data/raw/fr.txt"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for _ in range(30):
        print(next(f))

In [ ]:
# ============================================================
# CLEAN AND FILTER SENTENCES
# ============================================================

BRACKETS1 = re.compile(r"\[[^\]]*\]")
BRACKETS2 = re.compile(r"\([^)]*\)")
LETTER = re.compile(r"\p{L}")


def clean_sentence(text):
    text = text.strip()
    text = BRACKETS1.sub("", text)
    text = BRACKETS2.sub("", text)
    text = " ".join(text.split())

    if not text:
        return None

    if not LETTER.search(text):
        return None

    return text

In [ ]:
# ============================================================
# PROCESS FILES AND REMOVE DUPLICATES
# ============================================================

def process_file(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    seen = set()

    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:

        for i, line in enumerate(infile):
            line = line.strip()

            cleaned = clean_sentence(line)
            if cleaned is None:
                continue

            if cleaned not in seen:
                seen.add(cleaned)
                outfile.write(cleaned + "\n")

            if i % 100000 == 0:
                print(f"Processed {i:,} lines")

In [ ]:
# ============================================================
# CREATE AND VERIFY CLEANED DATASETS
# ============================================================

process_file(
    input_path="/content/drive/MyDrive/code_switch_project/data/raw/de.txt",
    output_path="/content/drive/MyDrive/code_switch_project/data/processed/de_clean.txt"
)

process_file(
    input_path="/content/drive/MyDrive/code_switch_project/data/raw/fr.txt",
    output_path="/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.txt"
)

# Preview cleaned German sentences
!head -n 20 "/content/drive/MyDrive/code_switch_project/data/processed/de_clean.txt"

# Preview cleaned French sentences
!head -n 20 "/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.txt"


In [ ]:
# ============================================================
# LOAD LABELED DATA
# ============================================================

def load_labeled(path, label):
    with open(path, "r", encoding="utf-8") as f:
        return [(line.strip(), label) for line in f if line.strip()]

In [ ]:
# ============================================================
# CONVERT CLEANED DATASETS TO CSV
# ============================================================

# German dataset
input_path = "/content/drive/MyDrive/code_switch_project/data/processed/de_clean.txt"
output_path = "/content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8", newline="") as fout:

    writer = csv.writer(fout)
    writer.writerow(["text", "label"])

    for line in fin:
        line = line.strip()
        if line:
            writer.writerow([line, "German"])

# French dataset
input_path = "/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.txt"
output_path = "/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8", newline="") as fout:

    writer = csv.writer(fout)
    writer.writerow(["text", "label"])

    for line in fin:
        line = line.strip()
        if line:
            writer.writerow([line, "French"])

In [ ]:
# ============================================================
# LOAD SWITCHLINGUA DATASET
# ============================================================

ds = load_dataset("Shelton1013/SwitchLingua_text")

In [ ]:
# ============================================================
# INSPECT SWITCHLINGUA DATASET
# ============================================================

print(f"Columns: {ds['train'].column_names}")
print(f"Total rows: {len(ds['train'])}")

print("\nFirst example:")
print(ds['train'][0])

In [ ]:
# ============================================================
# FILTER FRENCH-GERMAN EXAMPLES
# ============================================================

def is_french_german(example):
    first = example["first_language"].lower()
    second = example["second_language"].lower()

    # French-German or German-French
    is_fr_de = (first == "french" and second == "german") or \
               (first == "german" and second == "french")

    return is_fr_de


filtered_fr_de = ds["train"].filter(is_french_german)

print(f"French-German examples: {len(filtered_fr_de)}")


# Display one example

if len(filtered_fr_de) > 0:
    print("\nFirst example:")
    print(filtered_fr_de[0])

In [ ]:
# ============================================================
# EXTRACT FRENCH-GERMAN MIXED DATA
# ============================================================

fr_de_mixed = []

for example in filtered_fr_de:
    fr_de_mixed.append({
        "text": example["data_generation_result"],
        "label": "Mixed"
    })


# Convert to DataFrame

df_mixed = pd.DataFrame(fr_de_mixed)


# Save mixed dataset

output_path = "/content/drive/MyDrive/code_switch_project/data/raw/switchlingua_fr_de_mixed.csv"
df_mixed.to_csv(output_path, index=False)

print(f"Saved {len(df_mixed)} Mixed examples to {output_path}")


# Check extracted examples

print("\nFirst 3 examples:")
print(df_mixed.head(3))

print(type(df_mixed.iloc[0]["text"]))
print(df_mixed.iloc[0]["text"])

In [ ]:
# ============================================================
# CHECK DATASET SIZES
# ============================================================

!wc -l /content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv
!wc -l /content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv
!wc -l /content/drive/MyDrive/code_switch_project/data/raw/switchlingua_fr_de_mixed.csv

21212786 /content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv
42850589 /content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv
8512 /content/drive/MyDrive/code_switch_project/data/raw/switchlingua_fr_de_mixed.csv


## Initial Chat/Non-Chat Classification

An initial approach was used to identify chat-like sentences in the German, French, and mixed datasets. A zero-shot classifier was first used to classify samples as either `chat` or `not chat`. The resulting labels were then used to train a logistic regression classifier based on sentence embeddings.

This approach produced poor classification results and was therefore not used in the final dataset construction.

In [ ]:
# ============================================================
# CREATE 10K SAMPLES
# ============================================================

# German dataset
df_de = pd.read_csv("/content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv")
df_de_sample = df_de.sample(n=10000, random_state=42)
df_de_sample.to_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/de_sample_10k.csv",
    index=False
)


# French dataset
df_fr = pd.read_csv("/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv")
df_fr_sample = df_fr.sample(n=10000, random_state=42)
df_fr_sample.to_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_sample_10k.csv",
    index=False
)


# Mixed dataset
df_mix = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/raw/switchlingua_fr_de_mixed.csv"
)
df_mix_sample = df_mix.sample(n=min(8511, 10000), random_state=42)
df_mix_sample.to_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_sample_10k.csv",
    index=False
)

In [ ]:
# ============================================================
# INITIAL CHAT/NON-CHAT CLASSIFICATION
# ============================================================

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=0
)


def classify_sample(input_csv, output_csv):
    # Load CSV
    df = pd.read_csv(input_csv)

    # Safety check to prevent accidental full-dataset classification
    if len(df) > 20000:
        raise ValueError(
            f"Dataset too large ({len(df)} rows). "
            "Use a 10k sample file instead."
        )

    print(f"Loaded {len(df)} rows from {input_csv}")

    # Convert first column to text list
    texts = df.iloc[:, 0].astype(str).tolist()

    print("Starting classification...")

    # Run zero-shot classification
    results = classifier(
        texts,
        candidate_labels=["chat", "not chat"],
        multi_label=False,
        batch_size=64
    )

    # Extract labels and scores
    df["label"] = [r["labels"][0] for r in results]
    df["score"] = [r["scores"][0] for r in results]

    # Save output
    df.to_csv(output_csv, index=False)

In [ ]:
# ============================================================
# CLASSIFY GERMAN, FRENCH AND MIXED SAMPLES
# ============================================================

# German dataset
classify_sample(
    "/content/drive/MyDrive/code_switch_project/data/processed/de_sample_10k.csv",
    "/content/drive/MyDrive/code_switch_project/data/processed/de_sample_10k_labeled.csv"
)


# French dataset
classify_sample(
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_sample_10k.csv",
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_sample_10k_labeled.csv"
)


# Mixed dataset
classify_sample(
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_sample_10k.csv",
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_sample_10k_labeled.csv"
)

In [ ]:
# ============================================================
# COMBINE LABELED DATASETS
# ============================================================

de = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/de_sample_10k_labeled.csv"
)

fr = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_sample_10k_labeled.csv"
)

mix = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_sample_10k_labeled.csv"
)

df = pd.concat([de, fr, mix], ignore_index=True)

print(df.shape)
df.head()

In [ ]:
# ============================================================
# PREPARE CLASSIFICATION DATA
# ============================================================

df["label_num"] = df["label"].map({"chat": 1, "not chat": 0})

df.head()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# ============================================================
# SPLIT DATA INTO TRAINING, VALIDATION AND TEST SETS
# ============================================================

train, temp = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_num"]
)

val, test = train_test_split(
    temp,
    test_size=0.5,
    random_state=42,
    stratify=temp["label_num"]
)

len(train), len(val), len(test)

In [ ]:
# ============================================================
# GENERATE SENTENCE EMBEDDINGS
# ============================================================

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_train_emb = embedder.encode(
    train.iloc[:, 0].tolist(),
    batch_size=64,
    show_progress_bar=True
)

X_val_emb = embedder.encode(
    val.iloc[:, 0].tolist(),
    batch_size=64,
    show_progress_bar=True
)

X_test_emb = embedder.encode(
    test.iloc[:, 0].tolist(),
    batch_size=64,
    show_progress_bar=True
)

y_train = train["label_num"].tolist()
y_val = val["label_num"].tolist()
y_test = test["label_num"].tolist()

In [ ]:
# ============================================================
# TRAIN AND EVALUATE LOGISTIC REGRESSION
# ============================================================

clf = LogisticRegression(max_iter=2000)

clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)

print(classification_report(y_test, y_pred))

## Second Attempt: Full-Dataset Chat/Non-Chat Classification

The zero-shot classifier was then applied to the full German, French, and mixed datasets. The data was processed in chunks and saved incrementally to allow the process to resume after an interruption.

This approach was not used for the final dataset because the classification was too time-consuming and runtime interruptions made the process impractical.

In [ ]:
# ============================================================
# INITIALIZE CLASSIFIER AND PROGRESS DISPLAY
# ============================================================

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=0
)


def print_progress(current, total):
    bar_length = 30
    filled = int(bar_length * current / total)
    bar = "█" * filled + "░" * (bar_length - filled)
    percent = (current / total) * 100
    sys.stdout.write(f"\r[{bar}] {percent:.2f}% ({current}/{total} chunks)")
    sys.stdout.flush()

In [ ]:
# ============================================================
# CLASSIFY FULL DATASET IN CHUNKS
# ============================================================

def classify_full_dataset(input_csv, output_csv, chunk_size=500):
    # Count total rows to compute total chunks
    total_rows = sum(
        1 for _ in open(input_csv, "r", encoding="utf-8")
    ) - 1

    total_chunks = math.ceil(total_rows / chunk_size)

    # Resume from the last saved chunk if output already exists
    if os.path.exists(output_csv):
        processed_rows = sum(
            1 for _ in open(output_csv, "r", encoding="utf-8")
        ) - 1

        start_chunk = processed_rows // chunk_size

        print(
            f"Resuming from chunk {start_chunk} "
            f"({processed_rows} rows already saved)"
        )

        first_chunk = False

    else:
        processed_rows = 0
        start_chunk = 0

        print("Starting from the beginning")
        first_chunk = True


    reader = pd.read_csv(input_csv, chunksize=chunk_size)

    for chunk_idx, chunk in enumerate(reader):

        if chunk_idx < start_chunk:
            print_progress(chunk_idx, total_chunks)
            continue

        print(f"\nProcessing chunk {chunk_idx}...")

        texts = chunk.iloc[:, 0].astype(str).tolist()

        results = classifier(
            texts,
            candidate_labels=["chat", "not chat"],
            multi_label=False,
            batch_size=32
        )

        chunk["label"] = [r["labels"][0] for r in results]
        chunk["score"] = [r["scores"][0] for r in results]

        if first_chunk:
            chunk.to_csv(output_csv, index=False)
            first_chunk = False
        else:
            chunk.to_csv(
                output_csv,
                mode="a",
                header=False,
                index=False
            )

        print(f"Saved chunk {chunk_idx}")
        print_progress(chunk_idx + 1, total_chunks)

    print("\nDONE!")

In [ ]:
# ============================================================
# CLASSIFY GERMAN, FRENCH AND MIXED DATASETS
# ============================================================

# German dataset
classify_full_dataset(
    "/content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv",
    "/content/drive/MyDrive/code_switch_project/data/processed/de_full_labeled.csv"
)


# French dataset
classify_full_dataset(
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv",
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_full_labeled.csv"
)


# Mixed dataset
classify_full_dataset(
    "/content/drive/MyDrive/code_switch_project/data/raw/switchlingua_fr_de_mixed.csv",
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_full_labeled.csv"
)

## Final Approach: Chat/Non-Chat Classification

A high-confidence subset was first created using zero-shot classification. Only examples classified as `chat` or `non-chat` with a confidence of at least 0.90 were retained, producing balanced training datasets of 15,000 examples per class.

Separate multilingual classifiers were then fine-tuned for the French and German datasets. The trained classifiers were subsequently applied to the full datasets in batches, with the predictions saved as separate shards to allow processing to resume after interruptions.

In [3]:
# ============================================================
# HIGH-CONFIDENCE DATA MINING
# ============================================================

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=0,
    batch_size=64
)

LABELS = ["chat", "non-chat"]


def mine_dataset(
    input_csv,
    output_csv,
    text_col=0,
    chunk_size=500,
    threshold=0.95,
    target_chat=15000,
    target_nonchat=15000
):

    chat_count = 0
    nonchat_count = 0

    # Resume from an existing output file
    if os.path.exists(output_csv):
        df_existing = pd.read_csv(output_csv)
        chat_count = sum(df_existing["label"] == "chat")
        nonchat_count = sum(df_existing["label"] == "non-chat")
        first_write = False

        print(f"Resuming: chat={chat_count}, nonchat={nonchat_count}")

    else:
        first_write = True

    reader = pd.read_csv(input_csv, chunksize=chunk_size)

    for i, chunk in enumerate(reader):

        if chat_count >= target_chat and nonchat_count >= target_nonchat:
            print("\nTARGET REACHED → STOP")
            break

        texts = chunk["text"].astype(str).tolist()

        results = classifier(
            texts,
            candidate_labels=LABELS,
            multi_label=False
        )

        kept = []

        for text, r in zip(texts, results):

            # Get the confidence score for each class
            score_map = dict(zip(r["labels"], r["scores"]))
            chat_score = score_map["chat"]
            nonchat_score = score_map["non-chat"]

            if chat_score >= threshold and chat_count < target_chat:
                chat_count += 1
                kept.append([text, "chat", chat_score])

            elif nonchat_score >= threshold and nonchat_count < target_nonchat:
                nonchat_count += 1
                kept.append([text, "non-chat", nonchat_score])

        if kept:
            df_out = pd.DataFrame(
                kept,
                columns=["text", "label", "score"]
            )

            if first_write:
                df_out.to_csv(output_csv, index=False)
                first_write = False
            else:
                df_out.to_csv(
                    output_csv,
                    mode="a",
                    header=False,
                    index=False
                )

            os.sync()

        print(
            f"Chunk {i} | "
            f"chat={chat_count}/{target_chat} | "
            f"nonchat={nonchat_count}/{target_nonchat}"
        )

    print("\nDONE")
    print(f"Final: chat={chat_count}, nonchat={nonchat_count}")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [4]:
# ============================================================
# CHECK FRENCH DATASET
# ============================================================

df = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv",
    nrows=5
)

print(df.columns)
print(df.head())

Index(['text', 'label'], dtype='object')
                                            text   label
0                      En mémoire de Bine Monika  French
1               Eye 4 Films TDI Music présentent  French
2                                        L'ENFER  French
3  D'après la Divine Comédie de Dante Alighieri.  French
4                                        Avec...  French


In [5]:
# ============================================================
# CREATE FRENCH TRAINING DATASET
# ============================================================

mine_dataset(
    input_csv="/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv",
    output_csv="/content/drive/MyDrive/code_switch_project/data/processed/fr_train.csv",
    threshold=0.90,
    target_chat=15000,
    target_nonchat=15000
)

Resuming: chat=15000, nonchat=15000

TARGET REACHED → STOP

DONE
Final: chat=15000, nonchat=15000


In [6]:
# ============================================================
# CHECK FRENCH TRAINING DATASET
# ============================================================

df = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/fr_train.csv"
)

print(df.head(10))
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nLabels:")
print(df["label"].unique())

                                                text     label     score
0  Réalisé à partir des films d'archives provenan...  non-chat  0.946275
1  Dante voit ceux qui, lors de la grande rébelli...  non-chat  0.948728
2   Hommes d'État, Hommes de Science, cantonnés ici.  non-chat  0.985384
3  Ils n'ont aucune punition, mais ils n'ont aucu...  non-chat  0.978611
4  Chaque âme coupable comparaît devant le juge M...  non-chat  0.935582
5       Didon, Reine de Carthage et Hélène de Troie.  non-chat  0.929552
6         " de Lancelot et comment amour le saisit :  non-chat  0.909546
7  Le cercle des Gourmands gardé par le chien Cer...  non-chat  0.946325
8     Pluton surveille les avares et les dépensiers.  non-chat  0.953692
9  Les avares et gaspilleurs sont condamnés à dép...  non-chat  0.922273

Label counts:
label
non-chat    15000
chat        15000
Name: count, dtype: int64

Labels:
['non-chat' 'chat']


In [7]:
# ============================================================
# PREPARE FRENCH TRAINING DATA
# ============================================================

df = df[["text", "label"]]

df.to_csv("train_ready.csv", index=False)

label_map = {
    "chat": 0,
    "non-chat": 1
}

df["label"] = df["label"].map(label_map)


# Train/validation split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label"],
    random_state=42
)


# Convert to Hugging Face datasets

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]]
)

In [8]:
# ============================================================
# LOAD MULTILINGUAL MODEL
# ============================================================

model_name = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=128
    )


train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


# Dynamic padding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# Classification model

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ============================================================
# TRAIN FRENCH CHAT CLASSIFIER
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/code_switch_project/finetuned_chat_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.126849,0.103070,0.961667,0.961448
2,0.068508,0.118457,0.968000,0.967828
3,0.045270,0.123404,0.973000,0.972919


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2532, training_loss=0.10889641020158644, metrics={'train_runtime': 505.1633, 'train_samples_per_second': 160.344, 'train_steps_per_second': 5.012, 'total_flos': 701048520734112.0, 'train_loss': 0.10889641020158644, 'epoch': 3.0})

In [ ]:
# ============================================================
# SAVE AND EVALUATE FRENCH CLASSIFIER
# ============================================================

model_path = "/content/drive/MyDrive/code_switch_project/finetuned_chat_classifier"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Fine-tuning complete. Model saved.")

trainer.evaluate()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete. Model saved.


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.045270,0.123404,3,0.973000,0.972919


{'eval_loss': 0.12340404093265533,
 'eval_accuracy': 0.973,
 'eval_f1': 0.9729187562688064}

### Full French Dataset Classification

The fine-tuned French classifier was applied to the complete cleaned French dataset. The data was processed in chunks and stored as separate shard files so that the process could resume after an interrupted runtime.

In [ ]:
# ============================================================
# CLASSIFY THE FULL FRENCH DATASET
# ============================================================

MODEL_PATH = "/content/drive/MyDrive/code_switch_project/finetuned_chat_classifier"
INPUT_CSV_PATH = "/content/drive/MyDrive/code_switch_project/data/processed/fr_clean.csv"
SHARD_DIR = "/content/drive/MyDrive/code_switch_project/data/shards_fr"
FINAL_OUTPUT_CSV_PATH = "/content/drive/MyDrive/code_switch_project/data/classified_fr_output.csv"

CHUNK_SIZE = 20000
BATCH_SIZE = 512

os.makedirs(SHARD_DIR, exist_ok=True)


# Load the fine-tuned model and tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)

model.eval()

if device.type == "cuda":
    model = model.half()


# Resume from existing shard files if the process was interrupted

existing_shards = glob.glob(
    os.path.join(SHARD_DIR, "shard_*.csv")
)

start_chunk = len(existing_shards)
total_processed = start_chunk * CHUNK_SIZE

if start_chunk > 0:
    print(
        f"🔄 Found {start_chunk} existing shards. "
        f"Resuming from chunk {start_chunk}."
    )


reader = pd.read_csv(
    INPUT_CSV_PATH,
    chunksize=CHUNK_SIZE
)

label_inv_map = {
    0: "chat",
    1: "non-chat"
}


for i, chunk in enumerate(reader):

    if i < start_chunk:
        continue

    chunk["text"] = chunk["text"].fillna("").astype(str)
    texts = chunk["text"].tolist()

    all_preds, all_probs = [], []

    for j in range(0, len(texts), BATCH_SIZE):

        batch_texts = texts[j:j + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            max_length=128,
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            outputs = model(**inputs)

            probs = torch.softmax(
                outputs.logits,
                dim=1
            )

            preds = torch.argmax(
                probs,
                dim=1
            )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_probs.extend(
            probs.max(dim=1).values.cpu().numpy()
        )


    chunk["predicted_label"] = [
        label_inv_map[p]
        for p in all_preds
    ]

    chunk["confidence"] = all_probs


    # Save each processed chunk as a separate shard

    shard_path = os.path.join(
        SHARD_DIR,
        f"shard_{i:06d}.csv"
    )

    # Write to a temporary file before renaming it

    tmp_path = shard_path + ".tmp"

    chunk[
        ["text", "predicted_label", "confidence"]
    ].to_csv(
        tmp_path,
        index=False
    )

    os.rename(
        tmp_path,
        shard_path
    )

    total_processed += len(chunk)

    print(
        f"✅ Processed chunk {i+1}. "
        f"Total rows done: {total_processed:,}"
    )


print(
    f"🎉 INFERENCE COMPLETE! "
    f"Classified {total_processed:,} rows across "
    f"{i+1 - start_chunk if start_chunk else i+1} new shards."
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

🔄 Found 2143 existing shards. Resuming from chunk 2143.
🎉 INFERENCE COMPLETE! Classified 42,860,000 rows across 0 new shards.


In [ ]:
# ============================================================
# COMBINE FRENCH SHARDS
# ============================================================

shard_files = sorted(
    glob.glob(os.path.join(SHARD_DIR, "shard_*.csv"))
)

print(f"Combining {len(shard_files)} shards...")


with open(FINAL_OUTPUT_CSV_PATH, "wb") as outfile:

    for idx, shard_path in enumerate(shard_files):

        with open(shard_path, "rb") as infile:

            if idx == 0:
                outfile.write(infile.read())
            else:
                infile.readline()
                outfile.write(infile.read())


print(f"Done -> {FINAL_OUTPUT_CSV_PATH}")

Combining 2143 shards...
Done -> /content/drive/MyDrive/code_switch_project/data/classified_fr_output.csv


In [ ]:
# ============================================================
# CHECK FRENCH CLASSIFICATION RESULTS
# ============================================================

fr = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/classified_fr_output.csv"
)

print("FR chat:", (fr["predicted_label"] == "chat").sum())
print("FR non-chat:", (fr["predicted_label"] == "non-chat").sum())

FR chat: 13273051
FR non-chat: 29577537


### German Full-Dataset Classification

The same procedure was applied to the German dataset using a separately fine-tuned German chat classifier.

In [9]:
# ============================================================
# CHECK GERMAN DATASET
# ============================================================

df = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv",
    nrows=5
)

print(df.columns)
print(df.head())

Index(['text', 'label'], dtype='object')
                                    text   label
0   Ich geh lieber wieder an die Arbeit.  German
1  Verspielt nicht alle Streichhölzer...  German
2                          - Hallo, Mac.  German
3                        - Hallo, Click.  German
4                         Tag, zusammen.  German


In [10]:
# ============================================================
# CREATE GERMAN TRAINING DATASET
# ============================================================

mine_dataset(
    input_csv="/content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv",
    output_csv="/content/drive/MyDrive/code_switch_project/data/processed/de_train.csv",
    threshold=0.90,
    target_chat=15000,
    target_nonchat=15000
)

Resuming: chat=15000, nonchat=15000

TARGET REACHED → STOP

DONE
Final: chat=15000, nonchat=15000


In [13]:
# ============================================================
# CHECK GERMAN TRAINING DATASET
# ============================================================

df = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/de_train.csv",
)

print(df.head(10))

print(df["label"].value_counts())

                                    text     label     score
0   Ich geh lieber wieder an die Arbeit.  non-chat  0.919787
1                          - Hallo, Mac.      chat  0.956431
2                        - Hallo, Click.      chat  0.955465
3                         Tag, zusammen.      chat  0.983940
4                               - Hallo.      chat  0.953144
5       - Die kann ein Mann nicht essen.  non-chat  0.902695
6        - Er spricht mir aus der Seele.      chat  0.964606
7  Tagschicht, Nachtschicht, Tagschicht.  non-chat  0.951120
8                            Tanzen wir!  non-chat  0.942427
9                            Kommen Sie!      chat  0.934655
label
non-chat    15000
chat        15000
Name: count, dtype: int64


In [14]:
# ============================================================
# PREPARE GERMAN TRAINING DATA
# ============================================================

df = df[["text", "label"]]

df.to_csv("train_ready.csv", index=False)


# Map labels to integers

label_map = {
    "chat": 0,
    "non-chat": 1
}

df["label"] = df["label"].map(label_map)


# Train/validation split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label"],
    random_state=42
)


# Convert to Hugging Face datasets

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]]
)

In [15]:
# ============================================================
# LOAD GERMAN CLASSIFIER MODEL
# ============================================================

model_name = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=128
    )


train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)


# Dynamic padding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# Load classification model

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# ============================================================
# TRAIN GERMAN CHAT CLASSIFIER
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/code_switch_project/finetuned_de_chat_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


# Train

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.213324,0.159722,0.943000,0.944606
2,0.110567,0.127120,0.958667,0.959211
3,0.066417,0.148884,0.958333,0.958814


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2532, training_loss=0.15295291398938798, metrics={'train_runtime': 326.3337, 'train_samples_per_second': 248.212, 'train_steps_per_second': 7.759, 'total_flos': 669330857968416.0, 'train_loss': 0.15295291398938798, 'epoch': 3.0})

In [18]:
# ============================================================
# SAVE AND EVALUATE GERMAN CLASSIFIER
# ============================================================

model_path = "/content/drive/MyDrive/code_switch_project/finetuned_de_chat_classifier"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Fine-tuning complete. Model saved.")


# Evaluate the model

trainer.evaluate()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete. Model saved.


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.066417,0.127120,3,0.958667,0.959211


{'eval_loss': 0.12711994349956512,
 'eval_accuracy': 0.9586666666666667,
 'eval_f1': 0.9592105263157895}

In [ ]:
# ============================================================
# CLASSIFY THE FULL GERMAN DATASET
# ============================================================

MODEL_PATH = "/content/drive/MyDrive/code_switch_project/finetuned_de_chat_classifier"
INPUT_CSV_PATH = "/content/drive/MyDrive/code_switch_project/data/processed/de_clean.csv"
SHARD_DIR = "/content/drive/MyDrive/code_switch_project/data/shards_de"
FINAL_OUTPUT_CSV_PATH = "/content/drive/MyDrive/code_switch_project/data/classified_de_output.csv"

CHUNK_SIZE = 20000
BATCH_SIZE = 512

os.makedirs(SHARD_DIR, exist_ok=True)


# Load model and tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)

model.eval()

if device.type == "cuda":
    model = model.half()


# Resume from existing shard files if the process was interrupted

existing_shards = glob.glob(
    os.path.join(SHARD_DIR, "shard_*.csv")
)

start_chunk = len(existing_shards)
total_processed = start_chunk * CHUNK_SIZE

if start_chunk > 0:
    print(
        f"🔄 Found {start_chunk} existing shards. "
        f"Resuming from chunk {start_chunk}."
    )


reader = pd.read_csv(
    INPUT_CSV_PATH,
    chunksize=CHUNK_SIZE
)

label_inv_map = {
    0: "chat",
    1: "non-chat"
}


for i, chunk in enumerate(reader):

    if i < start_chunk:
        continue

    chunk["text"] = chunk["text"].fillna("").astype(str)
    texts = chunk["text"].tolist()

    all_preds, all_probs = [], []

    for j in range(0, len(texts), BATCH_SIZE):

        batch_texts = texts[j:j + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            max_length=128,
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            outputs = model(**inputs)

            probs = torch.softmax(
                outputs.logits,
                dim=1
            )

            preds = torch.argmax(
                probs,
                dim=1
            )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_probs.extend(
            probs.max(dim=1).values.cpu().numpy()

        )


    chunk["predicted_label"] = [
        label_inv_map[p]
        for p in all_preds
    ]

    chunk["confidence"] = all_probs


    # Save each processed chunk as a separate shard

    shard_path = os.path.join(
        SHARD_DIR,
        f"shard_{i:06d}.csv"
    )

    # Write to a temporary file before renaming it

    tmp_path = shard_path + ".tmp"

    chunk[
        ["text", "predicted_label", "confidence"]
    ].to_csv(
        tmp_path,
        index=False
    )

    os.rename(
        tmp_path,
        shard_path
    )

    total_processed += len(chunk)

    print(
        f"✅ Processed chunk {i+1}. "
        f"Total rows done: {total_processed:,}"
    )


print(
    f"🎉 INFERENCE COMPLETE! "
    f"Classified {total_processed:,} rows across "
    f"{i+1 - start_chunk if start_chunk else i+1} new shards."
)

In [ ]:
# ============================================================
# COMBINE GERMAN SHARDS
# ============================================================

FINAL_OUTPUT_CSV_PATH = "/content/drive/MyDrive/code_switch_project/data/processed/classified_de_output.csv"
SHARD_DIR = "/content/drive/MyDrive/code_switch_project/data/shards_de"

shard_files = sorted(
    glob.glob(
        os.path.join(SHARD_DIR, "shard_*.csv")
    )
)

print(f"Combining {len(shard_files)} shards...")


with open(FINAL_OUTPUT_CSV_PATH, "wb") as outfile:

    for idx, shard_path in enumerate(shard_files):

        with open(shard_path, "rb") as infile:

            if idx == 0:
                outfile.write(infile.read())
            else:
                infile.readline()
                outfile.write(infile.read())


print(f"Done -> {FINAL_OUTPUT_CSV_PATH}")

In [ ]:
# ============================================================
# CHECK GERMAN CLASSIFICATION RESULTS
# ============================================================

de = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/classified_de_output.csv"
)

print("DE chat:", (de["predicted_label"] == "chat").sum())
print("DE non-chat:", (de["predicted_label"] == "non-chat").sum())

DE chat: 7825007
DE non-chat: 13387778
['text', 'predicted_label', 'confidence']


In [ ]:
de["predicted_confidence"] = de.apply(
    lambda row: row["confidence"]
    if row["predicted_label"] == "non-chat"
    else 1 - row["confidence"],
    axis=1
)

print("Overall confidence:", de["predicted_confidence"].mean())

print("\nConfidence by predicted class:")
print(de.groupby("predicted_label")["predicted_confidence"].mean())

Overall confidence: 0.9310040666426388

Confidence by predicted class:
predicted_label
chat        0.915234
non-chat    0.940221
Name: predicted_confidence, dtype: float64


In [ ]:
# ============================================================
# CHECK MIXED DATASET
# ============================================================

mix = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_full_labeled.csv"
)

print(mix.head(3))
print(mix.columns.tolist())
print(mix.shape)

In [ ]:
# ============================================================
# CHECK MIXED DATASET LABEL DISTRIBUTION
# ============================================================

mix = pd.read_csv(
    "/content/drive/MyDrive/code_switch_project/data/processed/mix_full_labeled.csv"
)

print("Mix chat:", (mix["predicted_label"] == "chat").sum())
print("Mix non-chat:", (mix["predicted_label"] == "non-chat").sum())

In [ ]:
# ============================================================
# CHECK RESULTS
# ============================================================

de_path = "/content/drive/MyDrive/code_switch_project/data/processed/classified_de_output.csv"

de = pd.read_csv(de_path)

print("GERMAN")
print("Overall confidence:", de["confidence"].mean())
print("\nConfidence by predicted class:")
print(de.groupby("predicted_label")["confidence"].mean())

fr_path = "/content/drive/MyDrive/code_switch_project/data/processed/classified_fr_output.csv"

fr = pd.read_csv(fr_path)

print("\nFRENCH")
print("Overall confidence:", fr["confidence"].mean())
print("\nConfidence by predicted class:")
print(fr.groupby("predicted_label")["confidence"].mean())

print("\nConfidence values corrected and saved successfully.")

GERMAN
Overall confidence: 0.9310040666426388

Confidence by predicted class:
predicted_label
chat        0.915234
non-chat    0.940221
Name: confidence, dtype: float64

FRENCH
Overall confidence: 0.9718179621066148

Confidence by predicted class:
predicted_label
chat        0.956992
non-chat    0.978471
Name: confidence, dtype: float64

Confidence values corrected and saved successfully.
